In [ ]:
#in this notebook, I will implement a full modern GPT decoder only., with modern architectures:
#GQA,RoPe,SwiGLU,Weight Tying, Register_Buffer,Mixed Precision,Gradient_Accumulation.

In [ ]:
import torch,torch.nn as nn
import math
import torch.utils.checkpoint as checkpoint
from tokenizers import Tokenizer
import glob
import os
#importing the data
from google.colab import drive
drive.mount('/content/drive')
#class GPTConfig
class GPTConfig:
  def __init__(self,vocab_size,batch_size,seq_len,n_embd,n_kv_head,n_head,n_layer):
    self.vocab_size=vocab_size
    self.batch_size=batch_size
    self.seq_len=seq_len
    self.n_embd=n_embd
    self.n_kv_head=n_kv_head
    self.n_head=n_head
    self.n_layer=n_layer
#class Swiglu
class SwiGLU(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.n_embd = config.n_embd
    self.hidden_dim = 4 *self.n_embd
    self.gate_proj = nn.Linear(self.n_embd,self.hidden_dim)
    self.value_proj = nn.Linear(self.n_embd,self.hidden_dim)
    self.down_proj = nn.Linear(self.hidden_dim,self.n_embd)
  def forward(self,x):
    gate = torch.nn.functional.silu(self.gate_proj(x))
    value = self.value_proj(x)
    out = gate * value
    return self.down_proj(out)
#class RoPE
class RoPE(nn.Module):
  def __init__(self,config,base = 10000):
    super().__init__()
    self.n_embd = config.n_embd
    self.seq_len = config.seq_len
    self.n_head = config.n_head
    self.head_dim = self.n_embd // self.n_head
    inv_freq = 1.0 /(base**(torch.arange(0,self.head_dim,2)/self.head_dim))
    positions = torch.arange(self.seq_len)
    angels = torch.outer(positions,inv_freq)
    self.register_buffer("cos" ,torch.cos(angels))
    self.register_buffer("sin" ,torch.sin(angels))
  def forward(self,x):
    x1 = x[...,0::2]
    x2 = x[...,1::2]
    cos = self.cos[:x.shape[2]].unsqueeze(0).unsqueeze(0)
    sin = self.sin[:x.shape[2]].unsqueeze(0).unsqueeze(0)
    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos
    out = torch.empty_like(x)
    out[...,0::2] = rotated_x1
    out[...,1::2] = rotated_x2
    return out
#class GQA
class GQA(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.n_embd = config.n_embd
    self.batch_size = config.batch_size
    self.seq_len = config.seq_len
    self.n_head = config.n_head
    self.n_layer = config.n_layer
    self.n_kv_head = config.n_kv_head
    self.head_dim = self.n_embd // self.n_head
    self.n_rep = self.n_head // self.n_kv_head
    self.q_proj = nn.Linear(self.n_embd,self.n_head * self.head_dim)
    self.k_proj = nn.Linear(self.n_embd,self.n_kv_head * self.head_dim)
    self.v_proj = nn.Linear(self.n_embd,self.n_kv_head * self.head_dim)
    self.out_proj = nn.Linear(self.n_embd,self.n_embd)
    self.register_buffer("causal_mask",torch.tril(torch.ones(self.seq_len,self.seq_len)))
    self.rope = RoPE(config)
    self.weights_dropout = nn.Dropout(0.1)
    self.attn_out_dropout = nn.Dropout(0.1)
  def forward(self,x):
    self.batch_size,self.seq_len,_ = x.shape
    query = self.q_proj(x).reshape(self.batch_size,self.seq_len,self.n_head,self.head_dim).transpose(1,2)
    key = self.k_proj(x).reshape(self.batch_size,self.seq_len,self.n_kv_head,self.head_dim).transpose(1,2)
    value = self.v_proj(x).reshape(self.batch_size,self.seq_len,self.n_kv_head,self.head_dim).transpose(1,2)
    query = self.rope(query)
    key = self.rope(key)
    key = key.repeat_interleave(self.n_rep,dim=1)
    value = value.repeat_interleave(self.n_rep,dim=1)
    attention_scores = (query @ key.transpose(-1,-2)) / math.sqrt(self.head_dim)
    mask = self.causal_mask[:self.seq_len,:self.seq_len]
    attention_scores = attention_scores.masked_fill(mask==0,float("-inf"))
    weights = torch.softmax(attention_scores,dim=-1)
    weights = self.weights_dropout(weights)
    attention_output = weights @ value
    attention_output = attention_output.transpose(1,2).reshape(self.batch_size,self.seq_len,self.n_embd)
    attention_output = self.attn_out_dropout(attention_output)
    return self.out_proj(attention_output)
#class TransformerBlock
class TransformerBlock(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.ln1 = nn.RMSNorm(config.n_embd)
    self.attn = GQA(config)
    self.ln2 = nn.RMSNorm(config.n_embd)
    self.mlp = SwiGLU(config)
  def forward(self,x):
    original_x = x
    x = self.ln1(x)
    attn_output = self.attn(x)
    x = original_x + attn_output
    original_x = x
    x = self.ln2(x)
    mlp_output = self.mlp(x)
    x = original_x + mlp_output
    return x
#class TransformerStacks
class TransformerStack(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.n_layer = config.n_layer
    self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(self.n_layer)])
  def forward(self,x):
    for block in self.blocks:
      x = checkpoint.checkpoint(block,x,use_reentrant=False)
    return x
#class GPTModel
class GPTModel(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.vocab_size = config.vocab_size
    self.n_embd = config.n_embd
    self.embedding = nn.Embedding(self.vocab_size,self.n_embd)
    self.transformer = TransformerStack(config)
    self.final_ln = nn.RMSNorm(self.n_embd)
    self.lm_head = nn.Linear(self.n_embd,self.vocab_size)
    self.lm_head.weight = self.embedding.weight #weight tying
  def forward(self,token_ids):
    token_ids = self.embedding(token_ids)
    transformer_output = self.transformer(token_ids)
    transformer_output = self.final_ln(transformer_output)
    logits = self.lm_head(transformer_output)
    return logits
#Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = Tokenizer.from_pretrained("gpt2")
config = GPTConfig(vocab_size=tokenizer.get_vocab_size(),batch_size=8,seq_len=512,n_embd=256,n_kv_head=4,n_head=8,n_layer=7)
model = GPTModel(config)
model = model.to(device)
#needs explanation
def _init_weights(module, n_layer):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

model.apply(lambda m: _init_weights(m, config.n_layer))

# GPT-2 also scales residual-stream output projections by 1/sqrt(2*n_layer)
# to stop variance compounding across depth — apply to out_proj and down_proj:
for name, p in model.named_parameters():
    if name.endswith("out_proj.weight") or name.endswith("down_proj.weight"):
        nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
#needs explanation
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
scalar = torch.cuda.amp.GradScaler()
#Data Loading/Splitting ... etc
TOKEN_DIR = "/content/drive/MyDrive/"
TOKEN_FILE = f"{TOKEN_DIR}/all_tokens.pt"
DATA_SET = "/content/drive/MyDrive/datasets/TinyStories-train.txt"
DATA_CHUNK = 10_000_000
#total_read = 0
if not os.path.exists(TOKEN_FILE):
  chunk_name = 0
  with open(DATA_SET,"r",encoding="utf-8") as file:
    while True:
      text = file.read(DATA_CHUNK)
      if not text:
        break
      #total_read += len(text)
      chunk_name+=1
      token_ids = tokenizer.encode(text).ids
      tensor_ids = torch.tensor(token_ids,dtype=torch.long)
      torch.save(tensor_ids,f"{TOKEN_DIR}/TS_CHUNK{chunk_name:04d}.pt")
      del token_ids,tensor_ids
      print(f"CHUNK {chunk_name} Tokenized")
  token_ids = None
  for chunk in sorted(glob.glob(f"{TOKEN_DIR}/TS_CHUNK*.pt")):
    token = torch.load(chunk,map_location="cpu")
    if token_ids is None:
      token_ids = token
    else:
      token_ids = torch.cat([token_ids,token])
    del token
  torch.save(token_ids, TOKEN_FILE)
  print("Tokenization is Complete and Saved")
else:
  print("Tokenized dataset already exists. Skipping Tokenization.")
#loading the saved Tokenized Data
token_ids = torch.load(TOKEN_FILE)
print(f"Loaded {len(token_ids):,} tokens.")
#data splitting
train_end = int(len(token_ids)*0.7)
val_end = int(len(token_ids)*0.9)
train_data = token_ids[:train_end]
val_data = token_ids[train_end:val_end]
test_data = token_ids[val_end:]
#get_batch
def get_batch(token_ids,config):
  max_start = len(token_ids) - config.seq_len - 1
  start_indices = torch.randint(0,max_start,(config.batch_size,))
  inputs = torch.stack([token_ids[i:i+config.seq_len]for i in start_indices])
  targets = torch.stack([token_ids[i+1:i+config.seq_len+1]for i in start_indices])
  return inputs,targets

best_val_loss = float("inf")
checkpoint_path = "/content/drive/MyDrive/best_gpt_model.pt"
#estimate loss
def estimate_loss(data,n_loss_batches,config):
  loss_list = []
  model.eval()
  for i in range(n_loss_batches):
    with torch.no_grad():
      input_ids,target_ids = get_batch(data,config)
      input_ids = input_ids.to(device)
      target_ids= target_ids.to(device)
      logits = model(input_ids)
      loss = loss_fn(logits.view(-1,logits.shape[-1]),target_ids.view(-1))
      loss_list.append(loss.item())
  full_loss = sum(loss_list)/len(loss_list)
  model.train()
  return full_loss
#generation method
def generation(text,max_tokens,model,tokenizer):
  tokenized_text = tokenizer.encode(text).ids
  input_ids = torch.tensor(tokenized_text,dtype=torch.long).unsqueeze(0)
  input_ids = input_ids.to(device)
  with torch.no_grad():
    for _ in range(max_tokens):
      logits = model(input_ids)
      next_token = torch.argmax(logits[:,-1,:],dim=-1)
      next_token = next_token.unsqueeze(1)
      input_ids = torch.cat([input_ids, next_token], dim=-1)
  input_ids = input_ids.squeeze(0)
  generated_text = tokenizer.decode(input_ids.tolist())
  print(generated_text)
#Training loop
if not os.path.exists(checkpoint_path):

  accum_steps = 32
  model.train()
  optimizer.zero_grad()
  for step in range(45000):
    model.train()
    input_ids,target_ids = get_batch(train_data,config)
    input_ids = input_ids.to(device)
    target_ids = target_ids.to(device)
    with torch.autocast(device_type ="cuda",dtype=torch.float16):
      logits = model(input_ids)
      loss = loss_fn(logits.view(-1,logits.shape[-1]),target_ids.view(-1))
      loss_for_printing = loss
      loss = loss / accum_steps
    scalar.scale(loss).backward()
    if (step+1) % accum_steps == 0:
      scalar.step(optimizer)
      scalar.update()
      optimizer.zero_grad()
    if step % 1000 == 0:
      val_loss = estimate_loss(val_data,20,config)
      print(f"Step {step}, Train Loss {loss_for_printing.item():.4f},Val Loss {val_loss:.4f}")
      if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state_dict": model.state_dict(),
            "config": {
                "batch_size": config.batch_size,
                "vocab_size": config.vocab_size,
                "seq_len": config.seq_len,
                "n_layer": config.n_layer,
                "n_embd": config.n_embd,
                "n_head": config.n_head,
            },
            "step": step,
            "val_loss": best_val_loss,
        }, checkpoint_path)

        print(f"New best model saved! Val Loss: {best_val_loss:.4f}")
  #Test Data
  test_loss = estimate_loss(test_data,20,config)
  print(f"Test Loss: {test_loss}")
else:
  model.load_state_dict(torch.load(checkpoint_path)["model_state_dict"])
  print("checkpoint loaded, skipping training")
#generation
model.eval()
generation("my baby is",50,model,tokenizer)
generation("our dogs loves",50,model,tokenizer)
generation("the cat does",50,model,tokenizer)

Mounted at /content/drive


tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/tmp/ipykernel_513/3748809959.py:166: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scalar = torch.cuda.amp.GradScaler()


Tokenized dataset already exists. Skipping Tokenization.
Loaded 469,429,939 tokens.
checkpoint loaded, skipping training
my baby is very sick. She has to go to the hospital. She has to stay in the hospital for a long time. She is very sad.
She goes to the hospital. She sees a big dog. The dog is black and white. It has
our dogs loves to eat.
One day, Lily and Ben see a big dog. The dog is black and has a long tail. The dog is barking and running. Lily and Ben are scared. They run away from the dog.
"Help, help
the cat does not like it. It wants to eat the cheese. It wants to eat it. It wants to eat it. It wants to eat it. It wants to eat it.
"No, Tom, that is not a good idea. It is
